In [ ]:
import os
import numpy as np
import faiss
import time
from datetime import datetime
from sentence_transformers import SentenceTransformer
 
 
class LSFS:
    def __init__(self, root_dir="lsfs_storage"):
        self.root_dir = root_dir
        os.makedirs(self.root_dir, exist_ok=True)
 
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.dimension = 384
        self.index = faiss.IndexFlatL2(self.dimension)
 
        self.file_map = []
        self.version_store = {}
 
    # -------------------------
    # Create File
    # -------------------------
    def create_file(self, filename, content):
        path = os.path.join(self.root_dir, filename)
 
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
 
        self._index_file(filename, content)
        print("File created successfully.")
 
    # -------------------------
    # Index File
    # -------------------------
    def _index_file(self, filename, content):
        embedding = self.model.encode([content])
        vector = np.array(embedding).astype("float32")
 
        self.index.add(vector)
        self.file_map.append(filename)
 
    # -------------------------
    # Keyword Search
    # -------------------------
    def keyword_retrieve(self, keyword):
        start = time.time()
        results = []
 
        for file in os.listdir(self.root_dir):
            with open(os.path.join(self.root_dir, file), "r", encoding="utf-8") as f:
                if keyword.lower() in f.read().lower():
                    results.append(file)
 
        print("Keyword Time:", round(time.time() - start, 4), "seconds")
        return results
 
    # -------------------------
    # Semantic Search
    # -------------------------
    def semantic_retrieve(self, query, top_k=3):
 
        if len(self.file_map) == 0:
            return ["No files indexed"]
 
        query_embedding = self.model.encode([query])
        vector = np.array(query_embedding).astype("float32")
 
        k = min(top_k, len(self.file_map))
 
        D, I = self.index.search(vector, k)
 
        results = []
 
        for idx in I[0]:
            if idx != -1 and idx < len(self.file_map):
                results.append(self.file_map[idx])
 
        print("Total indexed files:", self.index.ntotal)
        return results
 
    # -------------------------
    # Overwrite + Version Save
    # -------------------------
    def overwrite(self, filename, new_content):
        path = os.path.join(self.root_dir, filename)
 
        if not os.path.exists(path):
            return "File not found."
 
        with open(path, "r", encoding="utf-8") as f:
            old_content = f.read()
 
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
 
        if filename not in self.version_store:
            self.version_store[filename] = []
 
        self.version_store[filename].append({
            "timestamp": timestamp,
            "content": old_content
        })
 
        with open(path, "w", encoding="utf-8") as f:
            f.write(new_content)
 
        return "File updated. Version saved."
 
    # -------------------------
    # Rollback
    # -------------------------
    def rollback(self, filename):
        if filename not in self.version_store or not self.version_store[filename]:
            return "No versions available."
 
        last_version = self.version_store[filename].pop()
        path = os.path.join(self.root_dir, filename)
 
        with open(path, "w", encoding="utf-8") as f:
            f.write(last_version["content"])
 
        return "Rollback successful."
 
    # -------------------------
    # Natural Language Parser
    # -------------------------
    def parse_command(self, command):
        command = command.lower()
 
        if "find" in command or "search" in command:
            return ("semantic", command)
 
        elif "keyword" in command:
            keyword = command.split()[-1]
            return ("keyword", keyword)
 
        elif "create" in command:
            return ("create", None)
 
        elif "overwrite" in command:
            return ("overwrite", None)
 
        elif "rollback" in command:
            return ("rollback", None)
 
        else:
            return ("unknown", None)
 
 
# -------------------------
# CLI Interface
# -------------------------
if __name__ == "__main__":
    fs = LSFS()
 
    while True:
        print("\n===== LSFS SYSTEM =====")
        print("1. Create File")
        print("2. Keyword Search")
        print("3. Semantic Search")
        print("4. Overwrite File")
        print("5. Rollback")
        print("6. Natural Language Command")
        print("7. Exit")
 
        choice = input("Enter choice: ")
 
        if choice == "1":
            name = input("Filename: ")
            content = input("Content: ")
            fs.create_file(name, content)
 
        elif choice == "2":
            keyword = input("Keyword: ")
            print(fs.keyword_retrieve(keyword))
 
        elif choice == "3":
            query = input("Semantic Query: ")
            print(fs.semantic_retrieve(query))
 
        elif choice == "4":
            name = input("Filename: ")
            content = input("New Content: ")
            print(fs.overwrite(name, content))
 
        elif choice == "5":
            name = input("Filename: ")
            print(fs.rollback(name))
 
        elif choice == "6":
            cmd = input("Enter command: ")
            action, data = fs.parse_command(cmd)
 
            if action == "semantic":
                print(fs.semantic_retrieve(data))
            elif action == "keyword":
                print(fs.keyword_retrieve(data))
            else:
            
                print("Command not recognized.")
 
        elif choice == "7":
            break